In [30]:
import json
import wandb
import random
from anthropic import Anthropic
from datasets import Dataset, ClassLabel
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
import numpy as np


In [14]:
DATASET_PATH = "../data/processed/dataset.json"
MODEL_NAME   = "microsoft/deberta-v3-small"
OUTPUT_DIR   = "model_finetuned/"
CATEGORIES   = ["billing", "technical", "shipping", "account", "general"]

CONFIG = {
    "model_name":  MODEL_NAME,
    "epochs":      5,
    "batch_size":  16,
    "lr":          2e-5,
    "max_length":  256,
    "train_ratio": 0.85,
    "seed":        42,
}

id2category = {i: label for i, label in enumerate(CATEGORIES)}
category2id = {label: i for i, label in enumerate(CATEGORIES)}

In [44]:
ds = Dataset.from_json(DATASET_PATH)
label_names = sorted(set(ds['category']))
label_feature = ClassLabel(names=label_names)
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {label: i for i, label in enumerate(label_names)}

ds = ds.cast_column('category', label_feature)
ds = ds.train_test_split(test_size=0.2, stratify_by_column='category',seed=42)

In [46]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'category', 'urgency', 'team'],
        num_rows: 23726
    })
    test: Dataset({
        features: ['text', 'category', 'urgency', 'team'],
        num_rows: 5932
    })
})